# 年龄分层受众价值聚类与画像预测（详细实施方案）
## 1. 业务目标

把年龄段划分为：高价值 / 中等价值 / 低价值，并给出可执行的投放动作（提价、保持、降价/排除）。

训练一个轻量“画像预测器”，当有新周期数据（或新广告组）到来时，能自动判断每个年龄段属于哪一类，并给出置信度。

输出CSV + 一页式 PDF 摘要，便于直接在 Google Ads 做年龄段出价调整（Bid Modifier）或排除。

## 2. 输入与输出
### 2.1 输入（仅用年龄报表）

必要字段：Age, Clicks, Impr., CTR, Conversions, Conv. rate, Avg CPC, Cost / conv., Cost

注意：若某年龄段出现 0 转化，也要保留该行（下游会做小样本保护）

### 2.2 交付物（落地件）

1. 聚类结果 CSV exports/age_clusters.csv
age_band, clicks, conv_rate, ctr, cpc, cpcv, value_score, cluster(H/M/L), confidence, action, bid_modifier, evidence

2. 预测器文件 models/age_persona_model.pkl（或 .json 规则版）

3. 建议动作表 exports/age_actions.csv
age_band, action(up/down/exclude/keep), bid_modifier(+10%/-30%/exclude), rationale

4. 一页式 PDF 摘要 reports/age_persona_summary.pdf 
        
        -年龄×价值热力图、各类占比、TOP 3 调整建议与预期影响

## 3. 方法总览（两阶段）
### 阶段 A：受众价值聚类（一次性/周期更新）

1. 指标标准化与小样本保护

--- 对 Conv. rate 和 CTR 用贝叶斯收缩（Empirical Bayes）做平滑：
cr_shrunk = (a0 + conversions) / (a0 + b0 + clicks)，默认先验强度 a0+b0=50（可配），避免小样本“极端好/差”误判。

--- 对成本类（CPC、Cost/conv.）取对数或做分位缩放，抑制长尾。

2. 价值评分（Value Score）

---推荐可乘型指数（可解释、可微调）：
value_score = (cr_shrunk / base_cr)^α 
            × (ctr_shrunk / base_ctr)^β 
            × (base_cpc / cpc)^γ 
            × (base_cpcv / cpcv)^δ

--- base_* 为全账户年龄加权基线；默认权重：α=0.5, β=0.2, γ=0.15, δ=0.15（可在配置里改）

--- 直觉：转化越高、点击质量越高、成本越低 ⇒ 得分越高

3. 加权聚类（K=3）

--- 用 features = [value_score, cr_shrunk, cpcv]（可加 ctr_shrunk, cpc），

--- 按 Clicks 作为样本权重做 k-means/k-medoids（k=3），避免小样本年龄段主导聚类中心。

--- 聚类后将 3 个簇按簇内中位 value_score排序并命名为 H/M/L。

4. 业务阈值微调 & map 到动作

-- 高价值(H)：默认 提价 +10%～+25%

-- 中价值(M)：默认 保持或小幅 ±5%

-- 低价值(L)：默认 降价 -15%～-40%；若 clicks ≥ min_clicks 且 cpcv≫基线，建议 排除

-- 小样本保护：clicks < min_clicks（默认 30）即使聚到 L，也先给“观察（keep）”，不立刻排除

5. 置信度（Confidence）

-- 组合两部分：

        a. cluster_margin：该样本至本簇中心与次优簇中心的距离差/比；

        b. data_strength：sqrt(clicks) 的归一化或 Wilson 下界/方差反比；

-- confidence = f(cluster_margin, data_strength)，0–1 输出到 CSV。

### 阶段 B：受众画像预测（持续自动化）

1. 标签：用阶段 A 的 H/M/L 作为目标变量（y）

2. 特征（x）：ctr_shrunk, cr_shrunk, cpc, cpcv, clicks（或加入 impr）

3. 模型：多项逻辑回归（可解释）或 梯度提升树（更稳定）

        -- 若样本点只有 6–7 个年龄段，可采用规则/阈值树（RuleFit 或 CART 深度≤2）生成明文规则便于复用

4. 验证：时间切分（若只有一次快照，用 bootstrap 评估稳定性）；输出准确率、宏平均 F1、稳定性指数（随时间的类标变动率）

5. 导出：

        -- 规则版：models/age_persona_rules.json（如：if cpcv <= 0.8*base and cr_shrunk >= 1.1*base then class=H）

        -- 模型版：age_persona_model.pkl（附标准化参数）

## 4. 计算与口径细节
### 4.1 基线与归一化
    a. base_cr、base_ctr、base_cpc、base_cpcv 取按 Clicks 加权的全龄段均值；导出到报告页顶部，便于复核。

    b. 置信带：同时给出 base_cpcv 的 [p10, p90] 分位，用来解释“显著高/低”。

### 4.2 小样本保护参数（可调）
    a. min_clicks = 30（低于此进入“观察”而非“排除/大幅降价”）

    b. 贝叶斯先验强度 prior_strength = 50；若数据量更少可增大。

    c. “极端高成本”判定：cpcv >= 1.8 × base_cpcv；“极端高转化”判定：cr_shrunk >= 1.3 × base_cr

### 4.3 动作映射（默认）
    a. H（高价值）：bid_modifier = +10% ~ +25%（按 value_score 分层：Top1 段 +25%，Top2 段 +15%，其余 +10%）

    b. M（中价值）：bid_modifier = -5% ~ +5%（基于与基线的差距给轻微调整）

    c. L（低价值）：bid_modifier = -15% ~ -40%；若 clicks ≥ min_clicks 且 cpcv ≥ 1.8×base_cpcv ⇒ exclude

    d. 所有动作都带上evidence（如：CR 1.22×base, CPCV 0.76×base, clicks=420）

In [ ]:
## 5. 交付数据结构（示例) exports/age_clusters.csv

age_band,clicks,conv_rate,ctr,cpc,cpcv,value_score,cluster,confidence,action,bid_modifier,evidence
18-24, 85, 0.9%, 0.8%, 2.10, 176.83, 0.42, L, 0.78, down, -35%, "CR 0.14×base; CPCV 11.7×base"
25-34, 210, 1.4%, 1.1%, 1.95, 60.86, 0.66, L, 0.72, down, -25%, "CPCV 4.0×base"
35-44, 320, 4.2%, 1.5%, 1.70, 22.10, 1.35, M, 0.64, keep, 0%, "CR≈base; CPCV≈base"
55-64, 280, 7.1%, 1.8%, 1.55, 12.45, 2.10, H, 0.81, up, +20%, "CR 1.3×base; CPCV 0.8×base"
65+,   190, 6.2%, 1.6%, 1.60, 10.90, 2.05, H, 0.79, up, +15%, "CR 1.2×base; CPCV 0.7×base"


In [ ]:
## exports/age_actions.csv
age_band,action,bid_modifier,rationale
18-24,exclude,1,"点击不少但转化极低且CPCV远高于基线; 不符合高意向"
25-34,down,-25%,"成本/转化高于基线4倍；先降价保留极少量曝光"
55-64,up,+20%,"转化率与CPCV全面优于基线; 建议加权扩量"

## 6. 可解释“预测器”两种形态
### 6.1 规则版（推荐在年龄段数量很少时）

1. 自动从聚类结果中抽取阈值规则（例如用 CART 深度≤2）：

    if cpcv <= 0.85*base_cpcv and cr_shrunk >= 1.1*base_cr then class=H

    elif cpcv >= 1.5*base_cpcv and cr_shrunk <= 0.9*base_cr then class=L

    else class=M

2. 输出 JSON，易读易改，稳定性高。

### 6.2 模型版（数据量更大或周期多时）

1. 多项逻辑回归 / LightGBM：输入标准化后的 4–5 个特征，输出 H/M/L 概率 + 置信度。

2. 交付 .pkl 文件 + 推理脚本（或在小程序内一键运行）。

## 7. 验收与效果评估（完全离线）
1. 一致性复核：CSV 中的 evidence 指标与原报表一致（容差 < 1e-6）。

2. 稳定性：对年龄段做 bootstrap（或按月滚动）看 H/M/L 变动率；低样本年龄段应更多落在 M（观察）。

3. 模拟收益：把 L 的 20% 花费挪到 H，估算“单位转化成本变化”和“预计转化量变化”（假设“边际不变”）——在 PDF 摘要中展示。

4. 回放对比：若你有多期年龄报表，做“训练 → 下一期预测”的回放，输出准确率/F1 与动作命中率（H 段提价后是否确实更优）。


## 8. 界面与操作（建议 Streamlit）

1. 数据上传：Age report PDF/CSV；显示解析成功的年龄段数量与时间范围

2. 参数：min_clicks、prior_strength、αβγδ、排除阈值（如 1.8×base_cpcv）

3. 结果页：

    -年龄×价值热力图；聚类散点（value_score vs cpcv）

    -“建议动作表”可在线微调；导出 age_actions.csv

4. 导出：age_clusters.csv、age_actions.csv、age_persona_summary.pdf、（可选）age_persona_model.pkl/json

5. 更新：下次上传新报表 → 一键重跑 → 预测器自动给出新标签与置信度

In [ ]:
## 9. 工程结构与依赖
age-persona/
  app.py
  core/
    loader.py             # 读 Age report，做列名映射、清洗
    smoothing.py          # 贝叶斯收缩/置信区间
    scoring.py            # value_score 与基线计算
    clustering.py         # 加权K=3聚类，命名H/M/L
    rules_or_model.py     # 规则抽取或模型训练与保存
    exporter.py           # CSV/PDF 导出
  config/
    thresholds.yaml       # min_clicks、αβγδ、排除阈值等
    mapping.yaml
  exports/ reports/ models/ logs/
  requirements.txt


#依赖：pandas、numpy、scikit-learn、matplotlib/altair、reportlab/weasyprint（PDF）、pdfplumber/camelot（如需解析 PDF 表格）。

## 10. 风险与缓解

1. 样本太少：使用 min_clicks 与贝叶斯收缩；统一落 M（观察）。

2. 指标异常（如 CPC、CPCV 的极端值）：采用分位裁剪（winsorize p1–p99）。

3. 一次性快照导致过拟合：保留规则版预测器以提高鲁棒性；每月更新时再迭代权重。